## Manual Vectorization

In [3]:
import jax
import jax.numpy as jnp

x = jnp.arange(5)
w = jnp.array([2., 3., 4.])

def convolve(x, w):
    output = []
    for i in range(1, len(x) - 1):
        output.append(jnp.dot(x[i-1:i+2], w))
    return jnp.array(output)

convolve(x, w)

Array([11., 20., 29.], dtype=float32)

In [5]:
xs = jnp.stack([x, x])
ws = jnp.stack([w, w])

def manually_batched_convolve(xs, ws):
    output = []
    for i in range(xs.shape[0]):
        output.append(convolve(xs[i],ws[i]))
    return jnp.stack(output)

manually_batched_convolve(xs,ws)

Array([[11., 20., 29.],
       [11., 20., 29.]], dtype=float32)

In [7]:
def manually_vectorized_convolve(xs, ws):
    output = []
    for i in range(1, xs.shape[-1]-1):
        o = jnp.sum(xs[:, i-1:i+2] * ws, axis=1)
        print(o)
        output.append(o)
    return jnp.stack(output, axis=1)

manually_vectorized_convolve(xs, ws)

[11. 11.]
[20. 20.]
[29. 29.]


Array([[11., 20., 29.],
       [11., 20., 29.]], dtype=float32)

## Automatic Vectorization

In [9]:
auto_batch_convolve = jax.vmap(convolve)

auto_batch_convolve(xs, ws)

Array([[11., 20., 29.],
       [11., 20., 29.]], dtype=float32)

In [10]:
??jax.vmap

Signature:
jax.vmap(
    fun: 'F',
    in_axes: 'int | None | Sequence[Any]' = 0,
    out_axes: 'Any' = 0,
    axis_name: 'AxisName | None' = None,
    axis_size: 'int | None' = None,
    spmd_axis_name: 'AxisName | tuple[AxisName, ...] | None' = None,
    sum_match: 'bool' = False,
) -> 'F'
Source:   
@partial(api_boundary, repro_api_name="jax.vmap")
def vmap(fun: F,
         in_axes: int | None | Sequence[Any] = 0,
         out_axes: Any = 0,
         axis_name: AxisName | None = None,
         axis_size: int | None = None,
         spmd_axis_name: AxisName | tuple[AxisName, ...] | None = None,
         sum_match: bool = False
         ) -> F:
  """Vectorizing map. Creates a function which maps ``fun`` over argument axes.

  Args:
    fun: Function to be mapped over additional axes.
    in_axes: An integer, None, or sequence of values specifying which input
      array axes to map over.

      If each positional argument to ``fun`` is an array, then ``in_axes`` can
      be an intege

In [12]:
jax.make_jaxpr(jax.vmap(convolve))(xs, ws)

{ lambda ; a:i32[2,5] b:f32[2,3]. let
    c:i32[2,3] = slice[limit_indices=(2, 3) start_indices=(0, 0) strides=None] a
    d:f32[2] = dot_general[
      dimension_numbers=(([1], [1]), ([0], [0]))
      preferred_element_type=float32
    ] c b
    e:i32[2,3] = slice[limit_indices=(2, 4) start_indices=(0, 1) strides=None] a
    f:f32[2] = dot_general[
      dimension_numbers=(([1], [1]), ([0], [0]))
      preferred_element_type=float32
    ] e b
    g:i32[2,3] = slice[limit_indices=(2, 5) start_indices=(0, 2) strides=None] a
    h:f32[2] = dot_general[
      dimension_numbers=(([1], [1]), ([0], [0]))
      preferred_element_type=float32
    ] g b
    i:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] d
    j:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] f
    k:f32[2,1] = broadcast_in_dim[broadcast_dimensions=(0,)] h
    l:f32[2,3] = concatenate[dimension=1] i j k
  in (l,) }

In [17]:
auto_batch_convolve_v2 = jax.vmap(convolve, in_axes=1, out_axes=1)

xst = jnp.transpose(xs)
wst = jnp.transpose(ws)

auto_batch_convolve_v2(xst, wst)

Array([[11., 11.],
       [20., 20.],
       [29., 29.]], dtype=float32)

In [23]:
batch_convolve_v3 = jax.vmap(convolve, in_axes=[0, None])

batch_convolve_v3(xs, w)

Array([[11., 20., 29.],
       [11., 20., 29.]], dtype=float32)

In [24]:
jitted_batch_convolve = jax.jit(auto_batch_convolve)

jitted_batch_convolve(xs, ws)

Array([[11., 20., 29.],
       [11., 20., 29.]], dtype=float32)

In [25]:
def pairwise(f, xs):
    return jax.vmap(lambda x: jax.vmap(lambda y: f(x, y))(xs))(xs)

In [35]:
from jax import random
key = random.key(0)

xs = jax.random.normal(key, (5, 5))
print(xs)

def dist(x, y):
    return jnp.sqrt(jnp.sum(x-y)**2)

pairwise(dist, xs)

[[ 1.6226422   2.0252647  -0.43359444 -0.07861735  0.1760909 ]
 [-0.97208923 -0.49529874  0.4943786   0.6643493  -0.9501635 ]
 [ 2.1795304  -1.9551506   0.35857072  0.15779513  1.2770847 ]
 [ 1.5104648   0.970656    0.59960806  0.0247007  -1.9164772 ]
 [-1.8593491   1.728144    0.04719035  0.814128    0.13132767]]


Array([[0.        , 4.570609  , 1.2939556 , 2.1228337 , 2.450345  ],
       [4.570609  , 0.        , 3.276654  , 2.4477758 , 2.1202645 ],
       [1.2939556 , 3.276654  , 0.        , 0.8288779 , 1.1563892 ],
       [2.1228337 , 2.4477758 , 0.8288779 , 0.        , 0.32751155],
       [2.450345  , 2.1202645 , 1.1563892 , 0.32751155, 0.        ]],      dtype=float32)

(5, 5, 4)